# Rveda Training Smoke Launcher

This notebook is a thin launcher for the Task 3.3 smoke run.

It installs the runtime dependencies, checks that the repo is visible, and then calls `train_grpo_smoke.py`.
It does not duplicate the training logic.


## 1. Install runtime dependencies

Run this once per Colab session.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

%pip install -q --upgrade pip
%pip install -q "openenv-core[core]>=0.2.3" datasets accelerate unsloth
print("Installed: openenv-core[core]>=0.2.3, datasets, accelerate, unsloth")

## 2. Confirm or clone the repo

> If the repo is not present in common Colab paths, this cell can clone it automatically.

The notebook expects `train_grpo_smoke.py` to be present in the current workspace or mounted folder.

In [ ]:
import os
import subprocess

repo_root = Path(os.environ.get("RVEDA_REPO_ROOT", Path.cwd()))
script_path = repo_root / "train_grpo_smoke.py"

if not script_path.exists():
    candidates = [
        Path("/content/rveda"),
        Path("/content/drive/MyDrive/rveda"),
        Path("/workspace/rveda"),
    ]
    for candidate in candidates:
        if (candidate / "train_grpo_smoke.py").exists():
            repo_root = candidate
            script_path = candidate / "train_grpo_smoke.py"
            break

if not script_path.exists():
    clone_target = Path("/content/rveda")
    clone_target.parent.mkdir(parents=True, exist_ok=True)
    clone_url = os.environ.get("RVEDA_GIT_URL", "https://github.com/<your-org>/rveda.git")
    print(f"Repo not found locally. Cloning from: {clone_url}")
    try:
        subprocess.check_call(["git", "clone", clone_url, str(clone_target)])
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Git clone failed. Set RVEDA_GIT_URL to your repo URL and rerun this cell."
        ) from exc
    repo_root = clone_target
    script_path = repo_root / "train_grpo_smoke.py"

if not script_path.exists():
    raise FileNotFoundError(
        "Could not find train_grpo_smoke.py after clone/lookup. "
        "Set RVEDA_REPO_ROOT to your repo path and rerun this cell."
    )

# Keep Colab checkout fresh so launcher uses latest fixes (including rollout_summary).
git_dir = repo_root / ".git"
if git_dir.exists():
    branch = os.environ.get("RVEDA_GIT_BRANCH", "")
    try:
        subprocess.check_call(["git", "-C", str(repo_root), "fetch", "--all", "--prune"])
        if branch:
            subprocess.check_call(["git", "-C", str(repo_root), "checkout", branch])
        subprocess.check_call(["git", "-C", str(repo_root), "pull", "--ff-only"])
    except subprocess.CalledProcessError as exc:
        print("Warning: git sync failed, continuing with local checkout.")
        print(exc)

os.environ["RVEDA_REPO_ROOT"] = str(repo_root.resolve())
%cd {repo_root}
print("Repo root:", repo_root)
print("Script path:", script_path)
print("RVEDA_REPO_ROOT:", os.environ["RVEDA_REPO_ROOT"])
if (repo_root / ".git").exists():
    rev = subprocess.check_output(["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True).strip()
    print("Repo commit:", rev)

## 3. Launch the smoke runner

This calls the existing script with a minimal Colab-friendly configuration.


In [ ]:
output_dir = repo_root / "artifacts" / "grpo_smoke_colab"
command = [
    sys.executable,
    str(script_path),
    "--model-name",
    "Qwen/Qwen2.5-7B-Instruct",
    "--output-dir",
    str(output_dir),
    "--task-ids",
    "v2_easy_overweight_schema_v1",
    "--samples-per-task",
    "1",
    "--episodes",
    "1",
    "--train-steps",
    "1",
    "--max-episode-steps",
    "2",
]
print("Running:", " ".join(command))
result = subprocess.run(command, text=True, capture_output=True)
print("\n--- stdout ---")
print(result.stdout or "<empty>")
print("\n--- stderr ---")
print(result.stderr or "<empty>")
if result.returncode != 0:
    raise RuntimeError(f"train_grpo_smoke.py failed with exit code {result.returncode}")

## 4. Inspect the generated artifacts


In [ ]:
summary_path = output_dir / "summary.json"
if not summary_path.exists():
    raise FileNotFoundError(f"Missing summary artifact: {summary_path}")

summary = json.loads(summary_path.read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2))
print("Artifacts:")
for file_name in ["scripted_baseline.json", "baseline_model_eval.json", "post_train_model_eval.json", "summary.json"]:
    print("-", output_dir / file_name)